# GENOMIRROR AI
## Evidence-Traceable Biomedical Knowledge Graph & Scientific Disagreement Explorer

**Goal:** Demonstrate how biomedical evidence records can be connected into a knowledge graph, screened for differing classifications, and checked for missing metadata.

**Important:** This notebook uses fictional synthetic examples. It is an educational prototype, not a diagnostic or clinical decision-support tool. A disagreement flag does not determine which classification is correct.

## 1. Install dependencies (if needed)

In [ ]:
# Uncomment this line in a fresh Google Colab/Jupyter environment if required:
# %pip install pandas numpy networkx matplotlib

## 2. Import libraries

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

OUTPUT_DIR = Path("genomirror_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("GENOMIRROR AI loaded.")
print("Output directory:", OUTPUT_DIR.resolve())

## 3. Create the demonstration dataset

In [ ]:
# Fictional synthetic records created only to demonstrate the workflow.
records = [
    {
        "record_id": "EV-001", "variant_id": "VAR-DEMO-001", "gene_id": "GENE-DEMO-A",
        "disease_id": "DISEASE-DEMO-X", "classification": "Pathogenic",
        "evidence_type": "functional_study", "source_name": "Synthetic Source Alpha",
        "source_url": "https://example.org/alpha", "publication_year": 2022,
        "review_status": "reviewed", "evidence_summary": "Synthetic functional evidence example."
    },
    {
        "record_id": "EV-002", "variant_id": "VAR-DEMO-001", "gene_id": "GENE-DEMO-A",
        "disease_id": "DISEASE-DEMO-X", "classification": "Uncertain significance",
        "evidence_type": "case_series", "source_name": "Synthetic Source Beta",
        "source_url": "https://example.org/beta", "publication_year": 2024,
        "review_status": "needs_review", "evidence_summary": "Synthetic record with a different example classification."
    },
    {
        "record_id": "EV-003", "variant_id": "VAR-DEMO-002", "gene_id": "GENE-DEMO-B",
        "disease_id": "DISEASE-DEMO-Y", "classification": "Benign",
        "evidence_type": "population_frequency", "source_name": "Synthetic Source Gamma",
        "source_url": "https://example.org/gamma", "publication_year": 2023,
        "review_status": "reviewed", "evidence_summary": "Synthetic population-frequency evidence example."
    },
    {
        "record_id": "EV-004", "variant_id": "VAR-DEMO-003", "gene_id": "GENE-DEMO-C",
        "disease_id": "DISEASE-DEMO-Z", "classification": "Uncertain significance",
        "evidence_type": None, "source_name": "Synthetic Source Delta",
        "source_url": None, "publication_year": None,
        "review_status": "needs_review", "evidence_summary": None
    },
    {
        "record_id": "EV-005", "variant_id": "VAR-DEMO-002", "gene_id": "GENE-DEMO-B",
        "disease_id": "DISEASE-DEMO-Y", "classification": "Benign",
        "evidence_type": "computational_prediction", "source_name": "Synthetic Source Epsilon",
        "source_url": "https://example.org/epsilon", "publication_year": 2025,
        "review_status": "reviewed", "evidence_summary": "Second synthetic record for the same example variant."
    },
    {
        "record_id": "EV-006", "variant_id": "VAR-DEMO-004", "gene_id": "GENE-DEMO-A",
        "disease_id": "DISEASE-DEMO-W", "classification": "Likely pathogenic",
        "evidence_type": "segregation", "source_name": "Synthetic Source Zeta",
        "source_url": "https://example.org/zeta", "publication_year": 2021,
        "review_status": "needs_review", "evidence_summary": "Synthetic segregation example."
    },
]

df = pd.DataFrame(records)
df

## 4. Validate schema and profile data quality

In [ ]:
REQUIRED_COLUMNS = [
    "record_id", "variant_id", "gene_id", "disease_id",
    "classification", "source_name", "review_status"
]
OPTIONAL_COLUMNS = [
    "evidence_type", "source_url", "publication_year", "evidence_summary"
]

missing_required = [column for column in REQUIRED_COLUMNS if column not in df.columns]
if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}")

for column in OPTIONAL_COLUMNS:
    if column not in df.columns:
        df[column] = pd.NA

df = df[REQUIRED_COLUMNS + OPTIONAL_COLUMNS].copy()

profile_rows = []
for column in df.columns:
    missing_mask = df[column].isna() | df[column].astype("string").str.strip().eq("")
    profile_rows.append({
        "column": column,
        "dtype": str(df[column].dtype),
        "missing_count": int(missing_mask.sum()),
        "missing_percent": round(float(missing_mask.mean() * 100), 1),
        "unique_values": int(df[column].nunique(dropna=True))
    })

quality_profile = pd.DataFrame(profile_rows)
display(quality_profile)

print(f"Records: {len(df)}")
print("Duplicate record IDs:", int(df["record_id"].duplicated().sum()))

## 5. Detect differing classifications

In [ ]:
# Rule-based screen: flags variants with more than one distinct non-empty classification.
# This does not decide which classification is correct.
disagreement_rows = []

for variant_id, group in df.groupby("variant_id", dropna=False):
    if pd.isna(variant_id) or not str(variant_id).strip():
        continue

    labels = sorted({
        str(value).strip()
        for value in group["classification"].dropna()
        if str(value).strip()
    })

    if len(labels) > 1:
        disagreement_rows.append({
            "variant_id": str(variant_id),
            "classification_count": len(labels),
            "classifications_seen": " | ".join(labels),
            "record_ids": " | ".join(group["record_id"].astype(str)),
            "source_names": " | ".join(group["source_name"].astype(str)),
            "recommended_next_step": "Review original records, dates, criteria, and context."
        })

disagreements = pd.DataFrame(disagreement_rows)
if disagreements.empty:
    print("No differing classifications found.")
else:
    display(disagreements)

## 6. Identify incomplete evidence records

In [ ]:
completeness_fields = [
    "source_url", "publication_year", "evidence_type",
    "evidence_summary", "review_status"
]
incomplete = df[["record_id", "variant_id", "source_name"] + completeness_fields].copy()

missing_mask = incomplete[completeness_fields].isna() | incomplete[completeness_fields].astype("string").apply(
    lambda column: column.str.strip().eq("")
)
incomplete["missing_field_count"] = missing_mask.sum(axis=1)
incomplete["missing_fields"] = missing_mask.apply(
    lambda row: ", ".join(row.index[row].tolist()), axis=1
)
incomplete["completeness_percent"] = (
    (1 - incomplete["missing_field_count"] / len(completeness_fields)) * 100
).round(1)

incomplete_records = incomplete[incomplete["missing_field_count"] > 0].sort_values(
    "missing_field_count", ascending=False
)
display(incomplete_records)

## 7. Build the biomedical knowledge graph

In [ ]:
G = nx.Graph()

for _, row in df.iterrows():
    variant = f"variant:{row['variant_id']}"
    gene = f"gene:{row['gene_id']}"
    disease = f"disease:{row['disease_id']}"
    evidence = f"evidence:{row['record_id']}"

    G.add_node(variant, node_type="variant", label=str(row["variant_id"]))
    G.add_node(gene, node_type="gene", label=str(row["gene_id"]))
    G.add_node(disease, node_type="disease", label=str(row["disease_id"]))
    G.add_node(
        evidence, node_type="evidence", label=str(row["record_id"]),
        classification=str(row["classification"]),
        source=str(row["source_name"])
    )

    G.add_edge(variant, gene, relation="associated_with_gene")
    G.add_edge(variant, disease, relation="associated_with_disease")
    G.add_edge(evidence, variant, relation="reports_variant")
    G.add_edge(evidence, gene, relation="mentions_gene")
    G.add_edge(evidence, disease, relation="mentions_disease")

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

## 8. Visualize the graph

In [ ]:
plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42, k=0.8)
node_types = nx.get_node_attributes(G, "node_type")
markers = {"variant": "o", "gene": "s", "disease": "^", "evidence": "D"}

for node_type, marker in markers.items():
    nodes = [node for node, kind in node_types.items() if kind == node_type]
    if nodes:
        nx.draw_networkx_nodes(
            G, pos, nodelist=nodes, node_shape=marker,
            node_size=850, alpha=0.9, label=node_type
        )

labels = {node: attrs.get("label", node) for node, attrs in G.nodes(data=True)}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=8)
nx.draw_networkx_edges(G, pos, alpha=0.35)
plt.title("GENOMIRROR AI — Synthetic Biomedical Evidence Graph")
plt.legend()
plt.axis("off")
plt.tight_layout()

graph_path = OUTPUT_DIR / "knowledge_graph.png"
plt.savefig(graph_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", graph_path)

## 9. Create a review queue

In [ ]:
review_items = []

for _, row in disagreements.iterrows():
    review_items.append({
        "priority": "High",
        "item_type": "classification_disagreement",
        "entity_id": row["variant_id"],
        "finding": f"Differing classifications: {row['classifications_seen']}",
        "next_step": row["recommended_next_step"]
    })

for _, row in incomplete_records.iterrows():
    review_items.append({
        "priority": "Medium",
        "item_type": "incomplete_evidence_record",
        "entity_id": row["record_id"],
        "finding": f"{int(row['missing_field_count'])} field(s) missing: {row['missing_fields']}",
        "next_step": "Verify the original source and fill missing metadata where possible."
    })

review_queue = pd.DataFrame(
    review_items,
    columns=["priority", "item_type", "entity_id", "finding", "next_step"]
)
display(review_queue)

## 10. Export results and summary report

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)

exports = {
    "evidence_records.csv": df,
    "data_quality_profile.csv": quality_profile,
    "classification_disagreements.csv": disagreements,
    "incomplete_evidence_records.csv": incomplete_records,
    "review_queue.csv": review_queue,
}

for filename, table in exports.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)

report = {
    "project": "GENOMIRROR AI",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "data_note": "Synthetic fictional records only; not real clinical evidence.",
    "record_count": int(len(df)),
    "graph_node_count": int(G.number_of_nodes()),
    "graph_edge_count": int(G.number_of_edges()),
    "disagreement_count": int(len(disagreements)),
    "incomplete_record_count": int(len(incomplete_records)),
    "review_queue_count": int(len(review_queue)),
    "limitations": [
        "Synthetic records are fictional examples.",
        "Disagreement flags do not determine which classification is correct.",
        "This project is not intended for clinical use.",
        "Real deployment requires verified sources, normalized identifiers, and expert review."
    ]
}

with open(OUTPUT_DIR / "summary_report.json", "w", encoding="utf-8") as file:
    json.dump(report, file, indent=2)

print(json.dumps(report, indent=2))
print("\nGenerated files:")
for file_path in sorted(OUTPUT_DIR.iterdir()):
    print("-", file_path.name)

## 11. Project limitations and future work

1. Replace synthetic examples with verified, source-linked public records.
2. Store source accessions, retrieval timestamps, versions, and original classification criteria.
3. Normalize variant and gene identifiers before comparing records.
4. Create a curated benchmark to evaluate disagreement detection.
5. Add an interactive web interface and domain-expert review workflow.

**Reminder:** This notebook is a learning prototype. Do not use its synthetic examples to make medical or clinical decisions.